In [1]:
pip install torch torchvision opencv-python

In [2]:
# Mount the Google Drive to Google Colab
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# defining a dataset

In [4]:
import torch
from torch.utils.data import Dataset
import cv2
import numpy as np

class VideoDataset(Dataset):

    def __init__(self, video_paths, labels, num_frames=16, img_size=224):
        self.video_paths = video_paths
        self.labels = labels
        self.num_frames = num_frames
        self.img_size = img_size

    def load_video(self, path):

        cap = cv2.VideoCapture(path)
        frames = []

        while len(frames) < self.num_frames:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.resize(frame, (self.img_size, self.img_size))
            frame = frame[:, :, ::-1]
            frames.append(frame)

        cap.release()

        frames = np.array(frames)

        if len(frames) < self.num_frames:
            pad = np.zeros((self.num_frames - len(frames), self.img_size, self.img_size, 3))
            frames = np.concatenate([frames, pad])

        frames = frames.transpose(3,0,1,2)

        return torch.tensor(frames, dtype=torch.float32)

    def __getitem__(self, idx):

        video = self.load_video(self.video_paths[idx])
        label = torch.tensor(self.labels[idx])

        return video, label

    def __len__(self):
        return len(self.video_paths)

In [5]:
import torch.nn as nn

class VideoPatchEmbedding(nn.Module):

    def __init__(self, img_size=224, patch_size=16, frames=16, embed_dim=768):

        super().__init__()

        self.proj = nn.Conv3d(
            in_channels=3,
            out_channels=embed_dim,
            kernel_size=(2, patch_size, patch_size),
            stride=(2, patch_size, patch_size)
        )

    def forward(self, x):

        x = self.proj(x)

        B, C, T, H, W = x.shape

        x = x.flatten(2)
        x = x.transpose(1,2)

        return x

In [6]:
class VideoTransformer(nn.Module):

    def __init__(
        self,
        num_classes=10,
        embed_dim=768,
        depth=6,
        heads=8,
        mlp_dim=1024
    ):
        super().__init__()

        self.patch_embed = VideoPatchEmbedding()

        self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))

        self.pos_embed = nn.Parameter(torch.randn(1,1000,embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=mlp_dim,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):

        B = x.shape[0]

        x = self.patch_embed(x)

        cls_tokens = self.cls_token.expand(B, -1, -1)

        x = torch.cat((cls_tokens, x), dim=1)

        x = x + self.pos_embed[:, :x.size(1)]

        x = self.transformer(x)

        x = self.norm(x)

        cls = x[:,0]

        return self.head(cls)

# Lables and Videos

In [14]:
# Labels need to be mapped to integers.

labels_txt = [
'PlayingCello',
'PlayingCello',
'PlayingCello',
'PlayingCello',
'PlayingCello',
'ShavingBeard',
'ShavingBeard',
'ShavingBeard',
'ShavingBeard',
'ShavingBeard',
'TennisSwing',
'TennisSwing',
'TennisSwing',
'TennisSwing',
'TennisSwing',
]

label_lookup = {
    'PlayingCello':0,
    'ShavingBeard':1,
    'TennisSwing':2,
}

labels = [label_lookup[e] for e in labels_txt]

In [15]:
import os

#os.listdir('/content/gdrive/MyDrive/Video_Classification_TestData/train_EvenFurtherReducedData')

video_dir = '/content/gdrive/MyDrive/Video_Classification_TestData/train_EvenFurtherReducedData'

video_paths = [os.path.join(video_dir,name) for name in os.listdir(video_dir)]



In [16]:
from torch.utils.data import DataLoader
import torch.optim as optim
import torch
import os

dataset = VideoDataset(video_paths, labels)

loader = DataLoader(dataset, batch_size=2, shuffle=True)

model = VideoTransformer(num_classes=3)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(10):

    for videos, targets in loader:

        videos = videos.to(device)
        targets = targets.to(device)

        outputs = model(videos)

        loss = criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

RuntimeError: The size of tensor a (1569) must match the size of tensor b (1000) at non-singleton dimension 1

# FIXME
Need to reconcile dimesions of video and model. See `VideoTransformer`

In [18]:
videos.shape

torch.Size([2, 3, 16, 224, 224])